# BatikCraft — LoRA Batik untuk Objek Apa Pun

Melatih **LoRA gaya batik** dari dataset batik saja. Wayang, hewan, bunga, kendaraan, dan objek lain tidak harus ada dalam dataset. Saat inference, bentuk berasal dari gambar objek; gaya berasal dari LoRA.

Pipeline inference memakai img2img, mask/alpha asli, ControlNet Canny opsional, motif referensi opsional, dan pemulihan outline. Output: `.safetensors`, PNG transparan, dan `.batikmodel` untuk BatikCraftStudio.


In [ ]:
import importlib, subprocess, sys
packages={
 'diffusers':'diffusers==0.39.0','transformers':'transformers>=4.48,<5',
 'accelerate':'accelerate>=1.2','peft':'peft>=0.17','datasets':'datasets>=3.2',
 'safetensors':'safetensors>=0.4','torchvision':'torchvision',
 'cv2':'opencv-python-headless>=4.10'}
missing=[]
for module,package in packages.items():
    try: importlib.import_module(module)
    except ImportError: missing.append(package)
if missing: subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path

@dataclass
class CFG:
    dataset_root:str=''  # kosong = folder gambar terbesar di /kaggle/input
    object_path:str=''   # contoh /kaggle/input/object-test/wayang.png
    mask_path:str=''
    motif_reference_path:str=''
    base_model:str='stable-diffusion-v1-5/stable-diffusion-v1-5'
    controlnet_model:str='lllyasviel/control_v11p_sd15_canny'
    output_dir:str='/kaggle/working/batikcraft-any-object-lora'
    trigger_word:str='bcr_batikstyle'
    model_id:str='batikcraft-style-any-object-v1'
    author:str='Balya Rochmadi'
    resolution:int=512
    max_images:int=0
    max_train_steps:int=1600
    batch_size:int=1
    grad_accum:int=4
    learning_rate:float=1e-4
    rank:int=16
    seed:int=2026
    lora_weight:float=.85
    strength:float=.38
    controlnet_scale:float=.90
    outline_strength:float=.65
    use_controlnet:bool=True
    low_vram:bool=False
cfg=CFG(); Path(cfg.output_dir).mkdir(parents=True,exist_ok=True)
print(asdict(cfg))


In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Accelerator Kaggle.')
print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))
print('VRAM GB',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))


## 1. Siapkan dataset gaya batik

Folder boleh memiliki subfolder `parang`, `kawung`, `ceplok`, `mega_mendung`, `floral`, dan lain-lain. Caption dibuat sebagai **style caption**, bukan kelas objek.


In [ ]:
import json,random,shutil
from pathlib import Path
from PIL import Image,ImageOps
EXT={'.png','.jpg','.jpeg','.webp','.bmp','.tif','.tiff'}
TAGS={'parang':'parang diagonal rhythm','kawung':'kawung geometric rosette',
'ceplok':'ceplok repeat','mega':'mega mendung cloud ornament','lereng':'lereng arrangement',
'floral':'floral tendril ornament','ornamen':'ornamental composition'}
def images(root): return sorted(p for p in Path(root).rglob('*') if p.is_file() and p.suffix.lower() in EXT)
def discover():
    choices=[(len(images(p)),p) for p in Path('/kaggle/input').iterdir() if p.is_dir()]
    choices=[x for x in choices if x[0]]
    if not choices: raise RuntimeError('Dataset gambar tidak ditemukan.')
    return max(choices)[1]
def caption(p):
    name='_'.join(x.lower().replace('-','_') for x in p.parts)
    tags=[v for k,v in TAGS.items() if k in name] or ['ornamental Indonesian batik composition']
    return f"{cfg.trigger_word}, authentic Indonesian batik textile style, {', '.join(tags)}, wax-resist canting linework, intricate isen-isen"
root=Path(cfg.dataset_root) if cfg.dataset_root else discover(); files=images(root)
random.Random(cfg.seed).shuffle(files)
if cfg.max_images>0: files=files[:cfg.max_images]
train=Path(cfg.output_dir)/'prepared'/'train'
if train.parent.exists(): shutil.rmtree(train.parent)
train.mkdir(parents=True)
with (train/'metadata.jsonl').open('w',encoding='utf-8') as h:
    for i,p in enumerate(files):
        out=train/f'{i:06d}.png'
        with Image.open(p) as im: ImageOps.fit(im.convert('RGB'),(cfg.resolution,cfg.resolution),Image.Resampling.LANCZOS).save(out)
        h.write(json.dumps({'file_name':out.name,'text':caption(p.relative_to(root))})+'\n')
print('dataset',root,'images',len(files),'train',train)


## 2. Train LoRA

Base model mempertahankan pengetahuan objek umum; LoRA hanya mengajarkan bahasa visual batik.


In [ ]:
import gc,math,json
import torch,torch.nn.functional as F
from accelerate import Accelerator
from datasets import load_dataset
from diffusers import AutoencoderKL,DDPMScheduler,StableDiffusionPipeline,UNet2DConditionModel
from diffusers.optimization import get_scheduler
from diffusers.utils import convert_state_dict_to_diffusers
from peft import LoraConfig
from peft.utils import get_peft_model_state_dict
from torch.utils.data import DataLoader
from torchvision import transforms
from transformers import CLIPTextModel,CLIPTokenizer

acc=Accelerator(gradient_accumulation_steps=cfg.grad_accum,mixed_precision='fp16')
tok=CLIPTokenizer.from_pretrained(cfg.base_model,subfolder='tokenizer')
text=CLIPTextModel.from_pretrained(cfg.base_model,subfolder='text_encoder')
vae=AutoencoderKL.from_pretrained(cfg.base_model,subfolder='vae')
unet=UNet2DConditionModel.from_pretrained(cfg.base_model,subfolder='unet')
noise=DDPMScheduler.from_pretrained(cfg.base_model,subfolder='scheduler')
vae.requires_grad_(False);text.requires_grad_(False);unet.requires_grad_(False)
unet.add_adapter(LoraConfig(r=cfg.rank,lora_alpha=cfg.rank,init_lora_weights='gaussian',target_modules=['to_k','to_q','to_v','to_out.0']))
unet.enable_gradient_checkpointing()
ds=load_dataset('imagefolder',data_dir=str(train.parent),split='train')
tf=transforms.Compose([transforms.Resize(cfg.resolution),transforms.CenterCrop(cfg.resolution),transforms.RandomHorizontalFlip(.1),transforms.ToTensor(),transforms.Normalize([.5],[.5])])
def prep(x):
    x['pixel_values']=[tf(im.convert('RGB')) for im in x['image']]
    x['input_ids']=tok(list(x['text']),max_length=tok.model_max_length,padding='max_length',truncation=True,return_tensors='pt').input_ids
    return x
ds=ds.with_transform(prep)
def collate(x): return {'pixel_values':torch.stack([e['pixel_values'] for e in x]).float(),'input_ids':torch.stack([e['input_ids'] for e in x])}
loader=DataLoader(ds,shuffle=True,collate_fn=collate,batch_size=cfg.batch_size,num_workers=2)
params=[p for p in unet.parameters() if p.requires_grad]
opt=torch.optim.AdamW(params,lr=cfg.learning_rate,weight_decay=1e-2)
updates=math.ceil(len(loader)/cfg.grad_accum); epochs=math.ceil(cfg.max_train_steps/max(updates,1))
lr=get_scheduler('constant_with_warmup',optimizer=opt,num_warmup_steps=min(100,cfg.max_train_steps//10),num_training_steps=cfg.max_train_steps)
unet,opt,loader,lr=acc.prepare(unet,opt,loader,lr); vae.to(acc.device,dtype=torch.float16); text.to(acc.device,dtype=torch.float16)
step=0;losses=[]
for epoch in range(epochs):
    unet.train()
    for b in loader:
        with acc.accumulate(unet):
            lat=vae.encode(b['pixel_values'].to(acc.device,dtype=torch.float16)).latent_dist.sample()*vae.config.scaling_factor
            eps=torch.randn_like(lat); t=torch.randint(0,noise.config.num_train_timesteps,(lat.shape[0],),device=lat.device).long()
            pred=unet(noise.add_noise(lat,eps,t),t,text(b['input_ids'].to(acc.device))[0]).sample
            target=noise.get_velocity(lat,eps,t) if noise.config.prediction_type=='v_prediction' else eps
            loss=F.mse_loss(pred.float(),target.float()); acc.backward(loss)
            if acc.sync_gradients: acc.clip_grad_norm_(params,1.0)
            opt.step();lr.step();opt.zero_grad(set_to_none=True)
        if acc.sync_gradients:
            step+=1;losses.append(float(loss.detach()))
            if step%20==0: print(step,cfg.max_train_steps,sum(losses[-20:])/len(losses[-20:]))
        if step>=cfg.max_train_steps: break
    if step>=cfg.max_train_steps: break
acc.wait_for_everyone(); lora_dir=Path(cfg.output_dir)/'lora';lora_dir.mkdir(exist_ok=True)
if acc.is_main_process:
    state=convert_state_dict_to_diffusers(get_peft_model_state_dict(acc.unwrap_model(unet)))
    StableDiffusionPipeline.save_lora_weights(str(lora_dir),unet_lora_layers=state,safe_serialization=True)
    (Path(cfg.output_dir)/'training-report.json').write_text(json.dumps({'steps':step,'samples':len(files),'mean_loss':sum(losses[-100:])/len(losses[-100:]),'object_dataset_required':False},indent=2))
lora_path=lora_dir/'pytorch_lora_weights.safetensors';print(lora_path)
del unet,vae,text,loader;gc.collect();torch.cuda.empty_cache()


## 3. Batikkan wayang atau objek lain

Isi `cfg.object_path`. PNG transparan terbaik. Pada gambar berlatar polos, mask diperkirakan dari warna sudut. Hasil akhir memakai alpha dan outline objek asli.


In [ ]:
import gc,cv2,numpy as np,torch
from PIL import Image,ImageChops,ImageFilter,ImageOps
from diffusers import AutoPipelineForImage2Image,ControlNetModel,StableDiffusionControlNetImg2ImgPipeline

def mask_of(src):
    a=src.getchannel('A')
    if a.getextrema()[0]<255:return a
    x=np.asarray(src.convert('RGB'),dtype=np.int16); bg=np.median(np.stack([x[0,0],x[0,-1],x[-1,0],x[-1,-1]]),axis=0)
    return Image.fromarray((np.sqrt(np.sum((x-bg)**2,axis=2))>=38).astype(np.uint8)*255).filter(ImageFilter.MaxFilter(3))
def square(src,mask):
    box=mask.getbbox(); c=src.crop(box);m=mask.crop(box);scale=min(cfg.resolution*.84/c.width,cfg.resolution*.84/c.height);size=(round(c.width*scale),round(c.height*scale));c=c.resize(size);m=m.resize(size)
    left=(cfg.resolution-size[0])//2;top=(cfg.resolution-size[1])//2; canvas=Image.new('RGB',(cfg.resolution,cfg.resolution),'white');canvas.paste(c.convert('RGB'),(left,top),m); sm=Image.new('L',canvas.size);sm.paste(m,(left,top));return canvas,sm,box
def motif_init(init,sm):
    if not cfg.motif_reference_path:return init
    with Image.open(cfg.motif_reference_path) as im: tile=ImageOps.fit(im.convert('RGB'),(128,128))
    tiled=Image.new('RGB',init.size,'white')
    for y in range(0,cfg.resolution,128):
        for x in range(-(64 if y//128%2 else 0),cfg.resolution,128):tiled.paste(tile,(x,y))
    mixed=Image.blend(init,tiled,.42);out=init.copy();out.paste(mixed,(0,0),sm);return out

def batify_object():
    with Image.open(cfg.object_path) as im: src=im.convert('RGBA')
    if cfg.mask_path:
        with Image.open(cfg.mask_path) as im: mask=im.convert('L').resize(src.size)
    else: mask=mask_of(src)
    init,sm,obox=square(src,mask);init=motif_init(init,sm)
    gray=cv2.cvtColor(np.asarray(init),cv2.COLOR_RGB2GRAY);edge=np.maximum(cv2.Canny(gray,70,160),cv2.Canny(np.asarray(sm),50,120));control=Image.fromarray(edge).convert('RGB')
    if cfg.use_controlnet:
        cn=ControlNetModel.from_pretrained(cfg.controlnet_model,torch_dtype=torch.float16)
        pipe=StableDiffusionControlNetImg2ImgPipeline.from_pretrained(cfg.base_model,controlnet=cn,torch_dtype=torch.float16,safety_checker=None,requires_safety_checker=False)
    else: pipe=AutoPipelineForImage2Image.from_pretrained(cfg.base_model,torch_dtype=torch.float16,safety_checker=None,requires_safety_checker=False)
    pipe.load_lora_weights(str(lora_path.parent),weight_name=lora_path.name,adapter_name='batik_style');pipe.set_adapters(['batik_style'],adapter_weights=[cfg.lora_weight])
    if cfg.low_vram: pipe.enable_model_cpu_offload()
    else: pipe.to('cuda')
    kw=dict(prompt=f'{cfg.trigger_word}, same isolated object transformed into authentic Indonesian batik ornament, preserve exact identity pose proportions and silhouette, wax-resist canting lines, isen-isen',negative_prompt='different object, changed silhouette, changed pose, extra object, photograph, background scene, text, watermark, blurry',image=init,strength=cfg.strength,num_inference_steps=28,guidance_scale=6.5,generator=torch.Generator('cuda').manual_seed(cfg.seed))
    if cfg.use_controlnet:kw.update(control_image=control,controlnet_conditioning_scale=cfg.controlnet_scale)
    raw=pipe(**kw).images[0];sb=sm.getbbox();target=(obox[2]-obox[0],obox[3]-obox[1]);crop=raw.convert('RGBA').crop(sb).resize(target);tm=mask.crop(obox).resize(target);crop.putalpha(tm)
    detail=ImageOps.autocontrast(src.crop(obox).resize(target).convert('L')).filter(ImageFilter.FIND_EDGES);sil=ImageChops.subtract(tm.filter(ImageFilter.MaxFilter(5)),tm.filter(ImageFilter.MinFilter(3)));e=ImageChops.multiply(ImageChops.lighter(detail,sil),tm).point(lambda v:round(v*cfg.outline_strength));line=Image.new('RGBA',target,(65,35,24,0));line.putalpha(e);crop.alpha_composite(line)
    out=Image.new('RGBA',src.size);out.alpha_composite(crop,(obox[0],obox[1]));folder=Path(cfg.output_dir)/'object-previews';folder.mkdir(exist_ok=True);result=folder/f'{Path(cfg.object_path).stem}-batik.png';out.save(result);init.save(folder/'init.png');control.save(folder/'control.png');raw.save(folder/'raw.png');del pipe;gc.collect();torch.cuda.empty_cache();return result
if cfg.object_path: object_result=batify_object();print(object_result)
else: object_result=None;print('Isi cfg.object_path untuk inference objek.')


## 4. Export `.batikmodel`


In [ ]:
import hashlib,json,zipfile
preview=[]
manifest={'format':'batikcraft-model-pack','schema_version':'1.0','model':{'id':cfg.model_id,'name':'BatikCraft Style LoRA — Any Object','version':'1.0.0','type':'lora','base_model_family':'sd15','trigger_words':[cfg.trigger_word],'recommended_weight':cfg.lora_weight,'resolution':cfg.resolution,'capabilities':['object','selection','style-transfer','unseen-object-batification'],'lora_file':'lora/pytorch_lora_weights.safetensors','author':cfg.author,'description':'Batik-only style LoRA for object-preserving img2img and ControlNet Batification.','license':'','controlnet_family':'canny','negative_prompt':'different object, changed silhouette, text, watermark, blurry','metadata':{'trainer':'batikcraft-any-object-lora-v1','object_dataset_required':False}},'files':[]}
files=[('lora/pytorch_lora_weights.safetensors','lora',lora_path.read_bytes())]
report=Path(cfg.output_dir)/'training-report.json'
if report.is_file():files.append(('training-report.json','training-report',report.read_bytes()))
manifest['files']=[{'path':p,'role':r,'sha256':hashlib.sha256(c).hexdigest(),'size':len(c)} for p,r,c in files]
batikmodel=Path(cfg.output_dir)/f'{cfg.model_id}.batikmodel'
with zipfile.ZipFile(batikmodel,'w',zipfile.ZIP_DEFLATED) as z:
    z.writestr('manifest.json',json.dumps(manifest,ensure_ascii=False,indent=2))
    for p,r,c in files:z.writestr(p,c)
print('LoRA',lora_path);print('BatikModel',batikmodel)


## Tuning

Bentuk berubah: `strength 0.25–0.34`, `controlnet_scale 0.95–1.15`, naikkan `outline_strength`, gunakan mask manual. Efek batik kurang kuat: naikkan `lora_weight` perlahan atau gunakan motif referensi.
